<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->

# Submit an uploaded Harbor wrapper

This companion to `harbor_taskset_e2e.ipynb` needs a running trusted subprocess platform with
`evaluator.harbor_agent_source_enabled: true`, Harbor and Docker. It uses the self-contained
wrapper beside this notebook, including a relative helper import and executable script.
The built-in Codex notebook requires no custom-source enablement.

Uploads and job results persist. Use fresh task names to avoid replacing existing entities.


In [ ]:
from pathlib import Path
from uuid import uuid4
import os

from nemo_evaluator.api.schemas import TasksetRef
from nemo_evaluator.sdk.harbor import inspect_agent_source, upload_harbor_dataset
from nemo_evaluator.sdk.resources import Evaluator
from nemo_evaluator_sdk.agent_eval.runtimes.harbor_runtime import HarborAgentTaskRunner, HarborRuntimeConfig
from nemo_platform_plugin.client.client import NemoClient
from nemo_platform_plugin.evaluator.client import EvaluatorClient

cwd = Path.cwd().resolve()
example = next(path for parent in (cwd, *cwd.parents)
               for path in (parent, parent / "plugins/nemo-evaluator/examples/harbor_taskset")
               if (path / "agent/harbor_wrapper.py").is_file())
workspace = os.environ.get("NMP_WORKSPACE", "default")
client = NemoClient(base_url=os.environ.get("NMP_BASE_URL", "http://localhost:8080"), workspace=workspace,
                    auth=os.environ.get("NMP_API_KEY"))
evaluator = Evaluator(EvaluatorClient.from_client(client))
name = f"uploaded-wrapper-{uuid4().hex[:8]}"


In [ ]:
inventory = inspect_agent_source(example / "agent")
print(f"Selected {len(inventory.entries)} entries ({inventory.total_bytes} bytes)")
print("Excluded:", inventory.excluded)


In [ ]:
receipt = upload_harbor_dataset(
    example / "harbor_dataset", client=client,
    fileset_ref=f"{workspace}/{name}", taskset_name=name, replace=False,
)
print(receipt.taskset_ref)


## Automatic upload during submission

The native runner below captures its local `agent_dir` before creating the job.


In [ ]:
runner = HarborAgentTaskRunner(config=HarborRuntimeConfig(
    jobs_dir=Path("./local-harbor-results"), agent_dir=example / "agent",
    agent_import_path="harbor_wrapper:WrappedAgent", n_concurrent_trials=1,
    agent_env={"EXAMPLE_MODE": "uploaded"}, agent_setup_timeout_multiplier=2.0,
))
job = evaluator.submit(tasks=TasksetRef(f"{workspace}/{name}"), target=runner)
print(job)


The wrapper has been captured and verified before submission. Later changes to the original
source or notebook working directory cannot change this job's source archive. Poll the existing
job with the companion notebook's status/result cells; do not rerun submission merely to poll.
The deliberate debug task reports a partial trial and error; the two normal tasks should pass.

Use `env_secrets={"NAME": SecretRef("workspace/secret")}` with matching `agent_env_from_host`
for credentials. Source filenames are filtered but not scanned for embedded secrets. Uploaded
archives are retained for explicit reuse and are not garbage-collected automatically.


## Upload once, then submit by Fileset reference

This alternative publishes the wrapped agent separately from job submission. The returned
source descriptor contains the exact `fileset_ref`, archive SHA-256, and format. Keep the
whole descriptor: a bare Fileset name does not pin the source bytes. You can reuse this
JSON in a later session or on a machine without the local `agent/` directory.

Running this section creates another job against the taskset uploaded above.


In [ ]:
from nemo_evaluator.sdk.harbor import upload_harbor_agent

agent_source = upload_harbor_agent(
    example / "agent",
    client=client,
    fileset_ref=f"{workspace}/{name}-agent-sources",
)
source_json = agent_source.model_dump_json(indent=2)
print(source_json)  # Save this descriptor to reuse the same uploaded archive later.


The next cell uses only the descriptor and stored taskset reference; it does not read or
upload local source. `agent_import_path` is relative to the uploaded source root.
The typed evaluator client exposes the explicit job-spec API used for this workflow.


In [ ]:
from nemo_evaluator.harbor.agent_source import HarborAgentSource
from nemo_evaluator.jobs.agent_spec import AgentEvalInputSpec, HarborRunnerTarget
from nemo_platform_plugin.client.types import RetryPolicy
from nemo_platform_plugin.evaluator.types import SubmitAgentEvalJobRequest

saved_source = HarborAgentSource.model_validate_json(source_json)
spec = AgentEvalInputSpec(
    tasks=receipt.taskset_ref,
    target=HarborRunnerTarget(
        agent_import_path="harbor_wrapper:WrappedAgent",
        agent_source=saved_source,
        n_concurrent_trials=1,
        agent_env={"EXAMPLE_MODE": "uploaded"},
        agent_setup_timeout_multiplier=2.0,
    ),
)
evaluator_client = EvaluatorClient.from_client(client)
# Do not automatically retry a POST whose response may be lost: it could create a second job.
referenced_job = evaluator_client.with_retry(RetryPolicy(max_retries=0)).submit_agent_eval_job(
    body=SubmitAgentEvalJobRequest(spec=spec.model_dump(mode="json")),
).data()
print(referenced_job.name)


In [ ]:
# Rerun this cell to check the existing job without submitting another one.
status = evaluator_client.get_agent_eval_job_status(name=referenced_job.name).data()
print(status)


As with automatic submission, the two normal tasks should pass and the deliberate debug
task should report an error/partial trial. A completed job status alone does not establish
that every trial succeeded; inspect the saved trials and scores using the companion notebook.
Reusing `source_json` keeps the same source checksum even if the local wrapper changes.
